<a href="https://colab.research.google.com/github/MuhammedAkkus/Colab_Muzik/blob/main/experiments/vocals/Sari_Tente_Vokal_Deneyi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys, subprocess, shutil
print("Python:", sys.version)
print("Free disk GB:", round(shutil.disk_usage("/content").free / 1e9, 1))
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "Free GPU not available; no paid fallback."

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Free disk GB: 207.5
Tesla T4, 15360 MiB



In [2]:
import subprocess
from pathlib import Path

ROOT = Path('/content/ACE-Step-1.5')
PIN = 'ca1e85fe9430179831e6bc6be790c332190a3866'
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ace-step/ACE-Step-1.5.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', '--detach', PIN], cwd=ROOT, check=True)
subprocess.run(['uv', 'sync', '--python', '3.12'], cwd=ROOT, check=True)
print('SETUP_OK: isolated Python 3.12, no paid API, no language model loaded.')


SETUP_OK: isolated Python 3.12, no paid API, no language model loaded.


In [3]:
import subprocess
from pathlib import Path
SOURCE = "\"\"\"Free, bounded ACE-Step Lego vocal experiment on the user's backing track.\"\"\"\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\n\nos.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\nos.environ['ACESTEP_PROJECT_ROOT'] = '/content/ACE-Step-1.5'\nos.environ['MPLBACKEND'] = 'Agg'\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\n\nROOT = Path('/content/ACE-Step-1.5')\nOUT = Path('/content/sari_tente_vokal')\nOUT.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), 'Free GPU required; no paid fallback.'\nprint('GPU:', torch.cuda.get_device_name(), flush=True)\n\nurl = 'https://raw.githubusercontent.com/MuhammedAkkus/Colab_Muzik/main/catalog/2026-10-05/preview.mp3'\nsource = OUT / 'original.mp3'\nurllib.request.urlretrieve(url, source)\nexpected = '6bdbb2ec59089210e8606f7029b0490b69eef859cafa66bdc571f3598154f795'\nassert hashlib.sha256(source.read_bytes()).hexdigest() == expected\nstart, duration = 16 * 4 * 60 / 104, 12 * 4 * 60 / 104\nbacking = OUT / 'backing.wav'\nsubprocess.run(['ffmpeg', '-y', '-i', str(source), '-ss', str(start),\n                '-t', str(duration), '-ar', '48000', str(backing)], check=True)\n\nprint('Downloading only VAE, text encoder and base music model.', flush=True)\ncheckpoints = ROOT / 'checkpoints'\nsnapshot_download('ACE-Step/Ace-Step1.5', local_dir=str(checkpoints),\n                  allow_patterns=['vae/*', 'Qwen3-Embedding-0.6B/*'])\nsnapshot_download('ACE-Step/acestep-v15-base',\n                  local_dir=str(checkpoints / 'acestep-v15-base'))\n\nfrom acestep import model_downloader\n# The generic preflight also requires unused Turbo and LM weights.\n# This audio-conditioned experiment only loads the two shared components.\nmodel_downloader.MAIN_MODEL_COMPONENTS = ['vae', 'Qwen3-Embedding-0.6B']\nfrom acestep.handler import AceStepHandler\nfrom acestep.llm_inference import LLMHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\nhandler = AceStepHandler()\nstatus, ok = handler.initialize_service(\n    project_root=str(ROOT), config_path='acestep-v15-base', device='cuda',\n    use_flash_attention=False, compile_model=False, offload_to_cpu=True,\n    offload_dit_to_cpu=False, quantization=None, prefer_source='huggingface')\nprint(status, flush=True)\nassert ok, status\nlyrics = '''[Chorus]\nBiz bu \\u015fehri yanl\\u0131\\u015f yerinden sevdik\nCebimizde birikmi\\u015f yar\\u0131m c\\u00fcmleler\n\\u00dc\\u015f\\u00fcyen ellerini cebine sakla\nKalan son \\u0131\\u015f\\u0131klar da s\\u00f6ner birazdan\n\n[Instrumental Outro]'''\nparams = GenerationParams(\n    task_type='lego', src_audio=str(backing),\n    instruction='Generate the VOCALS track based on the audio context:',\n    caption='An isolated expressive Turkish alternative pop lead vocal, warm natural male tenor, clear Turkish diction, intimate verse-like phrasing rising into an emotional melodic chorus. Follow the backing audio harmony and 104 BPM groove. Sing the four lines, then leave room for the instrumental turnaround. Dry close-mic vocal, no spoken narration, no instruments in this stem.',\n    lyrics=lyrics, vocal_language='tr', bpm=104, keyscale='B minor',\n    timesignature='4', duration=duration, inference_steps=50,\n    guidance_scale=7.0, seed=20261005, thinking=False,\n    use_cot_metas=False, use_cot_caption=False, use_cot_language=False,\n    repainting_start=0, repainting_end=duration, enable_normalization=False)\nconfig = GenerationConfig(batch_size=1, use_random_seed=False,\n                          seeds=[20261005], audio_format='wav')\nprint('GENERATING_VOCAL', flush=True)\nresult = generate_music(handler, LLMHandler(), params, config, save_dir=str(OUT))\nassert result.success and result.audios, result.error\nitem = result.audios[0]\nstem = Path(item['path'])\ndata, rate = sf.read(stem, always_2d=True)\nassert np.isfinite(data).all() and np.sqrt(np.mean(data ** 2)) > 1e-5\nassert abs(len(data) / rate - duration) < 1, 'Unexpected vocal length'\nsubprocess.run(['ffmpeg', '-y', '-i', str(stem), '-c:a', 'libmp3lame',\n                '-b:a', '192k', str(OUT / 'vocal_stem.mp3')], check=True)\nmix_filter = '[0:a]volume=0.65[b];[1:a]volume=0.8[v];[b][v]amix=inputs=2:normalize=0,loudnorm=I=-14:TP=-1.5:LRA=11'\nsubprocess.run(['ffmpeg', '-y', '-i', str(backing), '-i', str(stem),\n                '-filter_complex', mix_filter, '-c:a', 'libmp3lame',\n                '-b:a', '256k', str(OUT / 'Sari_Tente_vokal_deneyi.mp3')], check=True)\nreport = {'status': 'experimental_not_approved', 'cost_paid_api': 0,\n          'model': 'ACE-Step/acestep-v15-base', 'task': 'lego',\n          'source_sha256': expected, 'start_seconds': start, 'duration': duration,\n          'seed': 20261005, 'lyrics': lyrics, 'vocal_path': str(stem),\n          'vocal_rms': float(np.sqrt(np.mean(data ** 2))),\n          'note': 'Technical checks only; musical quality and diction require listening.'}\n(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('VOCAL_PROBE_OK', json.dumps(report, ensure_ascii=False), flush=True)\n"
with subprocess.Popen(["uv", "run", "--no-sync", "python", "-u", "-c", SOURCE], cwd="/content/ACE-Step-1.5", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
assert code == 0, "Vocal experiment failed; no paid fallback."
from IPython.display import Audio, display
display(Audio(filename="/content/sari_tente_vokal/Sari_Tente_vokal_deneyi.mp3"))
import os, subprocess, threading
from datetime import datetime
exec("\"\"\"Bound Colab compute use without purchasing resources or using paid APIs.\"\"\"\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nimport math\n\n\n@dataclass(frozen=True)\nclass ComputePolicy:\n    \"\"\"Approve one bounded run from a freshly observed resource-panel balance.\"\"\"\n\n    reserve_units: float = 5.0\n    max_run_seconds: int = 1800\n    max_observation_age_seconds: int = 600\n\n    def check(self, balance: float, hourly_rate: float,\n              observed_at: datetime, *, free_tier_verified: bool = False,\n              now: datetime | None = None) -> str:\n        \"\"\"Return the allowed mode, or fail closed for uncertain compute costs.\"\"\"\n        now = now or datetime.now(timezone.utc)\n        if observed_at.tzinfo is None or now.tzinfo is None:\n            raise ValueError('Resource observations must include a timezone.')\n        age = (now - observed_at).total_seconds()\n        if age < 0 or age > self.max_observation_age_seconds:\n            raise ValueError('Check the Colab resource panel again before running.')\n        if not all(math.isfinite(v) and v >= 0 for v in (balance, hourly_rate)):\n            raise ValueError('Unknown or invalid compute balance/rate.')\n        if balance == 0:\n            if not free_tier_verified:\n                raise ValueError('Verify free-tier access in the Colab resource panel.')\n            return 'free_quota'\n        estimate = hourly_rate * self.max_run_seconds / 3600\n        if hourly_rate == 0:\n            raise ValueError('A positive balance requires a verified usage rate.')\n        if balance - estimate <= self.reserve_units:\n            raise ValueError('Reserve reached: no new credit-consuming generation.')\n        return 'existing_credits_only'\n")
policy = ComputePolicy()
mode = policy.check(299.8, 1.15, datetime.fromisoformat("2026-10-05T09:05:59.993Z"))
os.environ["MUSIC_COMPUTE_MODE"] = mode
print("Compute mode:", mode, "Reserve: 5 units. No new purchases.")
SOURCE = "\"\"\"Bounded ACE-Step vocal experiment without new payments or paid APIs.\"\"\"\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\n\nos.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\nos.environ['ACESTEP_PROJECT_ROOT'] = '/content/ACE-Step-1.5'\nos.environ['MPLBACKEND'] = 'Agg'\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\n\nROOT = Path('/content/ACE-Step-1.5')\nOUT = Path('/content/sari_tente_vokal')\nOUT.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), 'CUDA GPU unavailable; no upgrade or paid fallback.'\nprint('GPU:', torch.cuda.get_device_name(), flush=True)\n\nurl = 'https://raw.githubusercontent.com/MuhammedAkkus/Colab_Muzik/main/catalog/2026-10-05/preview.mp3'\nsource = OUT / 'original.mp3'\nurllib.request.urlretrieve(url, source)\nexpected = '6bdbb2ec59089210e8606f7029b0490b69eef859cafa66bdc571f3598154f795'\nassert hashlib.sha256(source.read_bytes()).hexdigest() == expected\nstart, duration = 16 * 4 * 60 / 104, 12 * 4 * 60 / 104\nbacking = OUT / 'backing.wav'\nsubprocess.run(['ffmpeg', '-y', '-i', str(source), '-ss', str(start),\n                '-t', str(duration), '-ar', '48000', str(backing)], check=True)\n\nprint('Downloading only VAE, text encoder and base music model.', flush=True)\ncheckpoints = ROOT / 'checkpoints'\nsnapshot_download('ACE-Step/Ace-Step1.5', local_dir=str(checkpoints),\n                  allow_patterns=['vae/*', 'Qwen3-Embedding-0.6B/*'])\nsnapshot_download('ACE-Step/acestep-v15-base',\n                  local_dir=str(checkpoints / 'acestep-v15-base'))\n\nfrom acestep import model_downloader\n# The generic preflight also requires unused Turbo and LM weights.\n# This audio-conditioned experiment only loads the two shared components.\nmodel_downloader.MAIN_MODEL_COMPONENTS = ['vae', 'Qwen3-Embedding-0.6B']\nfrom acestep.handler import AceStepHandler\nfrom acestep.llm_inference import LLMHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\nhandler = AceStepHandler()\nstatus, ok = handler.initialize_service(\n    project_root=str(ROOT), config_path='acestep-v15-base', device='cuda',\n    use_flash_attention=False, compile_model=False, offload_to_cpu=True,\n    offload_dit_to_cpu=False, quantization=None, prefer_source='huggingface')\nprint(status, flush=True)\nassert ok, status\nlyrics = '''[Chorus]\nBiz bu \\u015fehri yanl\\u0131\\u015f yerinden sevdik\nCebimizde birikmi\\u015f yar\\u0131m c\\u00fcmleler\n\\u00dc\\u015f\\u00fcyen ellerini cebine sakla\nKalan son \\u0131\\u015f\\u0131klar da s\\u00f6ner birazdan\n\n[Instrumental Outro]'''\nparams = GenerationParams(\n    task_type='lego', src_audio=str(backing),\n    instruction='Generate the VOCALS track based on the audio context:',\n    caption='An isolated expressive Turkish alternative pop lead vocal, warm natural male tenor, clear Turkish diction, intimate verse-like phrasing rising into an emotional melodic chorus. Follow the backing audio harmony and 104 BPM groove. Sing the four lines, then leave room for the instrumental turnaround. Dry close-mic vocal, no spoken narration, no instruments in this stem.',\n    lyrics=lyrics, vocal_language='tr', bpm=104, keyscale='B minor',\n    timesignature='4', duration=duration, inference_steps=50,\n    guidance_scale=7.0, seed=20261005, thinking=False,\n    use_cot_metas=False, use_cot_caption=False, use_cot_language=False,\n    repainting_start=0, repainting_end=duration, enable_normalization=False)\nconfig = GenerationConfig(batch_size=1, use_random_seed=False,\n                          seeds=[20261005], audio_format='wav')\nprint('GENERATING_VOCAL', flush=True)\nresult = generate_music(handler, LLMHandler(), params, config, save_dir=str(OUT))\nassert result.success and result.audios, result.error\nitem = result.audios[0]\nstem = Path(item['path'])\ndata, rate = sf.read(stem, always_2d=True)\nassert np.isfinite(data).all() and np.sqrt(np.mean(data ** 2)) > 1e-5\nassert abs(len(data) / rate - duration) < 1, 'Unexpected vocal length'\nsubprocess.run(['ffmpeg', '-y', '-i', str(stem), '-c:a', 'libmp3lame',\n                '-b:a', '192k', str(OUT / 'vocal_stem.mp3')], check=True)\nmix_filter = '[0:a]volume=0.65[b];[1:a]volume=0.8[v];[b][v]amix=inputs=2:normalize=0,loudnorm=I=-14:TP=-1.5:LRA=11'\nsubprocess.run(['ffmpeg', '-y', '-i', str(backing), '-i', str(stem),\n                '-filter_complex', mix_filter, '-c:a', 'libmp3lame',\n                '-b:a', '256k', str(OUT / 'Sari_Tente_vokal_deneyi.mp3')], check=True)\nreport = {'status': 'experimental_not_approved', 'cost_paid_api': 0,\n          'compute_mode': os.environ.get('MUSIC_COMPUTE_MODE', 'unverified'),\n          'model': 'ACE-Step/acestep-v15-base', 'task': 'lego',\n          'source_sha256': expected, 'start_seconds': start, 'duration': duration,\n          'seed': 20261005, 'lyrics': lyrics, 'vocal_path': str(stem),\n          'vocal_rms': float(np.sqrt(np.mean(data ** 2))),\n          'note': 'Technical checks only; musical quality and diction require listening.'}\n(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('VOCAL_PROBE_OK', json.dumps(report, ensure_ascii=False), flush=True)\n"
with subprocess.Popen(["uv", "run", "--no-sync", "python", "-u", "-c", SOURCE], cwd="/content/ACE-Step-1.5", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    timer = threading.Timer(policy.max_run_seconds, proc.kill)
    timer.start()
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        code = proc.wait()
    finally:
        timer.cancel()
assert code == 0, "Vocal experiment failed; no paid fallback."
from IPython.display import Audio, display
display(Audio(filename="/content/sari_tente_vokal/Sari_Tente_vokal_deneyi.mp3"))
import subprocess
from pathlib import Path
SOURCE = "\"\"\"Free, bounded ACE-Step Lego vocal experiment on the user's backing track.\"\"\"\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\n\nos.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\nos.environ['ACESTEP_PROJECT_ROOT'] = '/content/ACE-Step-1.5'\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\n\nROOT = Path('/content/ACE-Step-1.5')\nOUT = Path('/content/sari_tente_vokal')\nOUT.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), 'Free GPU required; no paid fallback.'\nprint('GPU:', torch.cuda.get_device_name(), flush=True)\n\nurl = 'https://raw.githubusercontent.com/MuhammedAkkus/Colab_Muzik/main/catalog/2026-10-05/preview.mp3'\nsource = OUT / 'original.mp3'\nurllib.request.urlretrieve(url, source)\nexpected = '6bdbb2ec59089210e8606f7029b0490b69eef859cafa66bdc571f3598154f795'\nassert hashlib.sha256(source.read_bytes()).hexdigest() == expected\nstart, duration = 16 * 4 * 60 / 104, 12 * 4 * 60 / 104\nbacking = OUT / 'backing.wav'\nsubprocess.run(['ffmpeg', '-y', '-i', str(source), '-ss', str(start),\n                '-t', str(duration), '-ar', '48000', str(backing)], check=True)\n\nprint('Downloading only VAE, text encoder and base music model.', flush=True)\ncheckpoints = ROOT / 'checkpoints'\nsnapshot_download('ACE-Step/Ace-Step1.5', local_dir=str(checkpoints),\n                  allow_patterns=['vae/*', 'Qwen3-Embedding-0.6B/*'])\nsnapshot_download('ACE-Step/acestep-v15-base',\n                  local_dir=str(checkpoints / 'acestep-v15-base'))\n\nfrom acestep import model_downloader\n# The generic preflight also requires unused Turbo and LM weights.\n# This audio-conditioned experiment only loads the two shared components.\nmodel_downloader.MAIN_MODEL_COMPONENTS = ['vae', 'Qwen3-Embedding-0.6B']\nfrom acestep.handler import AceStepHandler\nfrom acestep.llm_inference import LLMHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\nhandler = AceStepHandler()\nstatus, ok = handler.initialize_service(\n    project_root=str(ROOT), config_path='acestep-v15-base', device='cuda',\n    use_flash_attention=False, compile_model=False, offload_to_cpu=True,\n    offload_dit_to_cpu=False, quantization=None, prefer_source='huggingface')\nprint(status, flush=True)\nassert ok, status\nlyrics = '''[Chorus]\nBiz bu \\u015fehri yanl\\u0131\\u015f yerinden sevdik\nCebimizde birikmi\\u015f yar\\u0131m c\\u00fcmleler\n\\u00dc\\u015f\\u00fcyen ellerini cebine sakla\nKalan son \\u0131\\u015f\\u0131klar da s\\u00f6ner birazdan\n\n[Instrumental Outro]'''\nparams = GenerationParams(\n    task_type='lego', src_audio=str(backing),\n    instruction='Generate the VOCALS track based on the audio context:',\n    caption='An isolated expressive Turkish alternative pop lead vocal, warm natural male tenor, clear Turkish diction, intimate verse-like phrasing rising into an emotional melodic chorus. Follow the backing audio harmony and 104 BPM groove. Sing the four lines, then leave room for the instrumental turnaround. Dry close-mic vocal, no spoken narration, no instruments in this stem.',\n    lyrics=lyrics, vocal_language='tr', bpm=104, keyscale='B minor',\n    timesignature='4', duration=duration, inference_steps=50,\n    guidance_scale=7.0, seed=20261005, thinking=False,\n    use_cot_metas=False, use_cot_caption=False, use_cot_language=False,\n    repainting_start=0, repainting_end=duration, enable_normalization=False)\nconfig = GenerationConfig(batch_size=1, use_random_seed=False,\n                          seeds=[20261005], audio_format='wav')\nprint('GENERATING_VOCAL', flush=True)\nresult = generate_music(handler, LLMHandler(), params, config, save_dir=str(OUT))\nassert result.success and result.audios, result.error\nitem = result.audios[0]\nstem = Path(item['path'])\ndata, rate = sf.read(stem, always_2d=True)\nassert np.isfinite(data).all() and np.sqrt(np.mean(data ** 2)) > 1e-5\nassert abs(len(data) / rate - duration) < 1, 'Unexpected vocal length'\nsubprocess.run(['ffmpeg', '-y', '-i', str(stem), '-c:a', 'libmp3lame',\n                '-b:a', '192k', str(OUT / 'vocal_stem.mp3')], check=True)\nmix_filter = '[0:a]volume=0.65[b];[1:a]volume=0.8[v];[b][v]amix=inputs=2:normalize=0,loudnorm=I=-14:TP=-1.5:LRA=11'\nsubprocess.run(['ffmpeg', '-y', '-i', str(backing), '-i', str(stem),\n                '-filter_complex', mix_filter, '-c:a', 'libmp3lame',\n                '-b:a', '256k', str(OUT / 'Sari_Tente_vokal_deneyi.mp3')], check=True)\nreport = {'status': 'experimental_not_approved', 'cost_paid_api': 0,\n          'model': 'ACE-Step/acestep-v15-base', 'task': 'lego',\n          'source_sha256': expected, 'start_seconds': start, 'duration': duration,\n          'seed': 20261005, 'lyrics': lyrics, 'vocal_path': str(stem),\n          'vocal_rms': float(np.sqrt(np.mean(data ** 2))),\n          'note': 'Technical checks only; musical quality and diction require listening.'}\n(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('VOCAL_PROBE_OK', json.dumps(report, ensure_ascii=False), flush=True)\n"
with subprocess.Popen(["uv", "run", "--no-sync", "python", "-u", "-c", SOURCE], cwd="/content/ACE-Step-1.5", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
assert code == 0, "Vocal experiment failed; no paid fallback."
from IPython.display import Audio, display
display(Audio(filename="/content/sari_tente_vokal/Sari_Tente_vokal_deneyi.mp3"))
import subprocess
from pathlib import Path
SOURCE = "\"\"\"Free, bounded ACE-Step Lego vocal experiment on the user's backing track.\"\"\"\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\n\nos.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\nos.environ['ACESTEP_PROJECT_ROOT'] = '/content/ACE-Step-1.5'\nos.environ['MPLBACKEND'] = 'Agg'\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\n\nROOT = Path('/content/ACE-Step-1.5')\nOUT = Path('/content/sari_tente_vokal')\nOUT.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), 'Free GPU required; no paid fallback.'\nprint('GPU:', torch.cuda.get_device_name(), flush=True)\n\nurl = 'https://raw.githubusercontent.com/MuhammedAkkus/Colab_Muzik/main/catalog/2026-10-05/preview.mp3'\nsource = OUT / 'original.mp3'\nurllib.request.urlretrieve(url, source)\nexpected = '6bdbb2ec59089210e8606f7029b0490b69eef859cafa66bdc571f3598154f795'\nassert hashlib.sha256(source.read_bytes()).hexdigest() == expected\nstart, duration = 16 * 4 * 60 / 104, 12 * 4 * 60 / 104\nbacking = OUT / 'backing.wav'\nsubprocess.run(['ffmpeg', '-y', '-i', str(source), '-ss', str(start),\n                '-t', str(duration), '-ar', '48000', str(backing)], check=True)\n\nprint('Downloading only VAE, text encoder and base music model.', flush=True)\ncheckpoints = ROOT / 'checkpoints'\nsnapshot_download('ACE-Step/Ace-Step1.5', local_dir=str(checkpoints),\n                  allow_patterns=['vae/*', 'Qwen3-Embedding-0.6B/*'])\nsnapshot_download('ACE-Step/acestep-v15-base',\n                  local_dir=str(checkpoints / 'acestep-v15-base'))\n\nfrom acestep import model_downloader\n# The generic preflight also requires unused Turbo and LM weights.\n# This audio-conditioned experiment only loads the two shared components.\nmodel_downloader.MAIN_MODEL_COMPONENTS = ['vae', 'Qwen3-Embedding-0.6B']\nfrom acestep.handler import AceStepHandler\nfrom acestep.llm_inference import LLMHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\nhandler = AceStepHandler()\nstatus, ok = handler.initialize_service(\n    project_root=str(ROOT), config_path='acestep-v15-base', device='cuda',\n    use_flash_attention=False, compile_model=False, offload_to_cpu=True,\n    offload_dit_to_cpu=False, quantization=None, prefer_source='huggingface')\nprint(status, flush=True)\nassert ok, status\nlyrics = '''[Chorus]\nBiz bu \\u015fehri yanl\\u0131\\u015f yerinden sevdik\nCebimizde birikmi\\u015f yar\\u0131m c\\u00fcmleler\n\\u00dc\\u015f\\u00fcyen ellerini cebine sakla\nKalan son \\u0131\\u015f\\u0131klar da s\\u00f6ner birazdan\n\n[Instrumental Outro]'''\nparams = GenerationParams(\n    task_type='lego', src_audio=str(backing),\n    instruction='Generate the VOCALS track based on the audio context:',\n    caption='An isolated expressive Turkish alternative pop lead vocal, warm natural male tenor, clear Turkish diction, intimate verse-like phrasing rising into an emotional melodic chorus. Follow the backing audio harmony and 104 BPM groove. Sing the four lines, then leave room for the instrumental turnaround. Dry close-mic vocal, no spoken narration, no instruments in this stem.',\n    lyrics=lyrics, vocal_language='tr', bpm=104, keyscale='B minor',\n    timesignature='4', duration=duration, inference_steps=50,\n    guidance_scale=7.0, seed=20261005, thinking=False,\n    use_cot_metas=False, use_cot_caption=False, use_cot_language=False,\n    repainting_start=0, repainting_end=duration, enable_normalization=False)\nconfig = GenerationConfig(batch_size=1, use_random_seed=False,\n                          seeds=[20261005], audio_format='wav')\nprint('GENERATING_VOCAL', flush=True)\nresult = generate_music(handler, LLMHandler(), params, config, save_dir=str(OUT))\nassert result.success and result.audios, result.error\nitem = result.audios[0]\nstem = Path(item['path'])\ndata, rate = sf.read(stem, always_2d=True)\nassert np.isfinite(data).all() and np.sqrt(np.mean(data ** 2)) > 1e-5\nassert abs(len(data) / rate - duration) < 1, 'Unexpected vocal length'\nsubprocess.run(['ffmpeg', '-y', '-i', str(stem), '-c:a', 'libmp3lame',\n                '-b:a', '192k', str(OUT / 'vocal_stem.mp3')], check=True)\nmix_filter = '[0:a]volume=0.65[b];[1:a]volume=0.8[v];[b][v]amix=inputs=2:normalize=0,loudnorm=I=-14:TP=-1.5:LRA=11'\nsubprocess.run(['ffmpeg', '-y', '-i', str(backing), '-i', str(stem),\n                '-filter_complex', mix_filter, '-c:a', 'libmp3lame',\n                '-b:a', '256k', str(OUT / 'Sari_Tente_vokal_deneyi.mp3')], check=True)\nreport = {'status': 'experimental_not_approved', 'cost_paid_api': 0,\n          'model': 'ACE-Step/acestep-v15-base', 'task': 'lego',\n          'source_sha256': expected, 'start_seconds': start, 'duration': duration,\n          'seed': 20261005, 'lyrics': lyrics, 'vocal_path': str(stem),\n          'vocal_rms': float(np.sqrt(np.mean(data ** 2))),\n          'note': 'Technical checks only; musical quality and diction require listening.'}\n(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('VOCAL_PROBE_OK', json.dumps(report, ensure_ascii=False), flush=True)\n"
with subprocess.Popen(["uv", "run", "--no-sync", "python", "-u", "-c", SOURCE], cwd="/content/ACE-Step-1.5", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
assert code == 0, "Vocal experiment failed; no paid fallback."
from IPython.display import Audio, display
display(Audio(filename="/content/sari_tente_vokal/Sari_Tente_vokal_deneyi.mp3"))


GPU: Tesla T4
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable

AssertionError: Vocal experiment failed; no paid fallback.

In [4]:
import os, subprocess, threading
from datetime import datetime
exec("\"\"\"Bound Colab compute use without purchasing resources or using paid APIs.\"\"\"\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nimport math\n\n\n@dataclass(frozen=True)\nclass ComputePolicy:\n    \"\"\"Approve one bounded run from a freshly observed resource-panel balance.\"\"\"\n\n    reserve_units: float = 5.0\n    max_run_seconds: int = 1800\n    max_observation_age_seconds: int = 600\n\n    def check(self, balance: float, hourly_rate: float,\n              observed_at: datetime, *, free_tier_verified: bool = False,\n              now: datetime | None = None) -> str:\n        \"\"\"Return the allowed mode, or fail closed for uncertain compute costs.\"\"\"\n        now = now or datetime.now(timezone.utc)\n        if observed_at.tzinfo is None or now.tzinfo is None:\n            raise ValueError('Resource observations must include a timezone.')\n        age = (now - observed_at).total_seconds()\n        if age < 0 or age > self.max_observation_age_seconds:\n            raise ValueError('Check the Colab resource panel again before running.')\n        if not all(math.isfinite(v) and v >= 0 for v in (balance, hourly_rate)):\n            raise ValueError('Unknown or invalid compute balance/rate.')\n        if balance == 0:\n            if not free_tier_verified:\n                raise ValueError('Verify free-tier access in the Colab resource panel.')\n            return 'free_quota'\n        estimate = hourly_rate * self.max_run_seconds / 3600\n        if hourly_rate == 0:\n            raise ValueError('A positive balance requires a verified usage rate.')\n        if balance - estimate <= self.reserve_units:\n            raise ValueError('Reserve reached: no new credit-consuming generation.')\n        return 'existing_credits_only'\n")
policy = ComputePolicy()
mode = policy.check(299.8, 1.15, datetime.fromisoformat("2026-10-05T09:09:18.285Z"))
os.environ["MUSIC_COMPUTE_MODE"] = mode
print("Compute mode:", mode, "Reserve: 5 units. No new purchases.")
SOURCE = "\"\"\"Bounded ACE-Step vocal experiment without new payments or paid APIs.\"\"\"\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\n\nos.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\nos.environ['ACESTEP_PROJECT_ROOT'] = '/content/ACE-Step-1.5'\nos.environ['MPLBACKEND'] = 'Agg'\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\n\nROOT = Path('/content/ACE-Step-1.5')\nOUT = Path('/content/sari_tente_vokal')\nOUT.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), 'CUDA GPU unavailable; no upgrade or paid fallback.'\nprint('GPU:', torch.cuda.get_device_name(), flush=True)\n\nurl = 'https://raw.githubusercontent.com/MuhammedAkkus/Colab_Muzik/main/catalog/2026-10-05/preview.mp3'\nsource = OUT / 'original.mp3'\nurllib.request.urlretrieve(url, source)\nexpected = '6bdbb2ec59089210e8606f7029b0490b69eef859cafa66bdc571f3598154f795'\nassert hashlib.sha256(source.read_bytes()).hexdigest() == expected\nstart, duration = 16 * 4 * 60 / 104, 12 * 4 * 60 / 104\nbacking = OUT / 'backing.wav'\nsubprocess.run(['ffmpeg', '-y', '-i', str(source), '-ss', str(start),\n                '-t', str(duration), '-ar', '48000', str(backing)], check=True)\n\nprint('Downloading only VAE, text encoder and base music model.', flush=True)\ncheckpoints = ROOT / 'checkpoints'\nsnapshot_download('ACE-Step/Ace-Step1.5', local_dir=str(checkpoints),\n                  allow_patterns=['vae/*', 'Qwen3-Embedding-0.6B/*'])\nsnapshot_download('ACE-Step/acestep-v15-base',\n                  local_dir=str(checkpoints / 'acestep-v15-base'))\n\nfrom acestep import model_downloader\n# The generic preflight also requires unused Turbo and LM weights.\n# This audio-conditioned experiment only loads the two shared components.\nmodel_downloader.MAIN_MODEL_COMPONENTS = ['vae', 'Qwen3-Embedding-0.6B']\nfrom acestep.handler import AceStepHandler\nfrom acestep.llm_inference import LLMHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\nhandler = AceStepHandler()\nstatus, ok = handler.initialize_service(\n    project_root=str(ROOT), config_path='acestep-v15-base', device='cuda',\n    use_flash_attention=False, compile_model=False, offload_to_cpu=True,\n    offload_dit_to_cpu=False, quantization=None, prefer_source='huggingface')\nprint(status, flush=True)\nassert ok, status\n# T4 float16 overflowed even with eager attention on this Base/Lego task.\n# The release suggests ACESTEP_DTYPE, but does not read it in initialization.\nhandler.dtype = torch.float32\nhandler.model = handler.model.float()\nhandler.vae = handler.vae.float()\nhandler.text_encoder = handler.text_encoder.float()\nhandler.silence_latent = handler.silence_latent.float()\nprint('T4 stability mode: float32, CPU offload for VAE/text encoder.', flush=True)\nlyrics = '''[Chorus]\nBiz bu \\u015fehri yanl\\u0131\\u015f yerinden sevdik\nCebimizde birikmi\\u015f yar\\u0131m c\\u00fcmleler\n\\u00dc\\u015f\\u00fcyen ellerini cebine sakla\nKalan son \\u0131\\u015f\\u0131klar da s\\u00f6ner birazdan\n\n[Instrumental Outro]'''\nparams = GenerationParams(\n    task_type='lego', src_audio=str(backing),\n    instruction='Generate the VOCALS track based on the audio context:',\n    caption='An isolated expressive Turkish alternative pop lead vocal, warm natural male tenor, clear Turkish diction, intimate verse-like phrasing rising into an emotional melodic chorus. Follow the backing audio harmony and 104 BPM groove. Sing the four lines, then leave room for the instrumental turnaround. Dry close-mic vocal, no spoken narration, no instruments in this stem.',\n    lyrics=lyrics, vocal_language='tr', bpm=104, keyscale='B minor',\n    timesignature='4', duration=duration, inference_steps=50,\n    guidance_scale=7.0, seed=20261005, thinking=False,\n    use_cot_metas=False, use_cot_caption=False, use_cot_language=False,\n    repainting_start=0, repainting_end=duration, enable_normalization=False)\nconfig = GenerationConfig(batch_size=1, use_random_seed=False,\n                          seeds=[20261005], audio_format='wav')\nprint('GENERATING_VOCAL', flush=True)\nresult = generate_music(handler, LLMHandler(), params, config, save_dir=str(OUT))\nassert result.success and result.audios, result.error\nitem = result.audios[0]\nstem = Path(item['path'])\ndata, rate = sf.read(stem, always_2d=True)\nassert np.isfinite(data).all() and np.sqrt(np.mean(data ** 2)) > 1e-5\nassert abs(len(data) / rate - duration) < 1, 'Unexpected vocal length'\nsubprocess.run(['ffmpeg', '-y', '-i', str(stem), '-c:a', 'libmp3lame',\n                '-b:a', '192k', str(OUT / 'vocal_stem.mp3')], check=True)\nmix_filter = '[0:a]volume=0.65[b];[1:a]volume=0.8[v];[b][v]amix=inputs=2:normalize=0,loudnorm=I=-14:TP=-1.5:LRA=11'\nsubprocess.run(['ffmpeg', '-y', '-i', str(backing), '-i', str(stem),\n                '-filter_complex', mix_filter, '-c:a', 'libmp3lame',\n                '-b:a', '256k', str(OUT / 'Sari_Tente_vokal_deneyi.mp3')], check=True)\nreport = {'status': 'experimental_not_approved', 'cost_paid_api': 0,\n          'compute_mode': os.environ.get('MUSIC_COMPUTE_MODE', 'unverified'),\n          'model': 'ACE-Step/acestep-v15-base', 'task': 'lego',\n          'source_sha256': expected, 'start_seconds': start, 'duration': duration,\n          'seed': 20261005, 'lyrics': lyrics, 'vocal_path': str(stem),\n          'vocal_rms': float(np.sqrt(np.mean(data ** 2))),\n          'note': 'Technical checks only; musical quality and diction require listening.'}\n(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('VOCAL_PROBE_OK', json.dumps(report, ensure_ascii=False), flush=True)\n"
with subprocess.Popen(["/content/ACE-Step-1.5/.venv/bin/python", "-u", "-c", SOURCE], cwd="/content/ACE-Step-1.5", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    timer = threading.Timer(policy.max_run_seconds, proc.kill)
    timer.start()
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        code = proc.wait()
    finally:
        timer.cancel()
assert code == 0, "Vocal experiment failed; no paid fallback."
from IPython.display import Audio, display
display(Audio(filename="/content/sari_tente_vokal/Sari_Tente_vokal_deneyi.mp3"))


Compute mode: existing_credits_only Reserve: 5 units. No new purchases.
GPU: Tesla T4
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --en

In [5]:
import json, zipfile
from pathlib import Path
from google.colab import files
root = Path("/content/sari_tente_vokal")
report = json.loads((root / "report.json").read_text())
with zipfile.ZipFile("/content/Sari_Tente_vokal_deneyi.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [root / "Sari_Tente_vokal_deneyi.mp3", root / "vocal_stem.mp3", root / "backing.wav", root / "report.json", Path(report["vocal_path"])]:
        archive.write(path, path.name)
files.download("/content/Sari_Tente_vokal_deneyi.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import subprocess
subprocess.run(["uv", "pip", "install", "--python", "/content/ACE-Step-1.5/.venv/bin/python", "faster-whisper"], check=True, timeout=180)
CHECK = "\"\"\"Independent local ASR check, not a musical-quality approval.\"\"\"\nimport json\nfrom pathlib import Path\nfrom faster_whisper import WhisperModel\n\nroot = Path('/content/sari_tente_vokal')\nreport = json.loads((root / 'report.json').read_text())\nmodel = WhisperModel('medium', device='cpu', compute_type='int8', cpu_threads=2)\nsegments, info = model.transcribe(\n    report['vocal_path'], language='tr', beam_size=5,\n    vad_filter=True, condition_on_previous_text=False,\n)\nobserved = [{'start': s.start, 'end': s.end, 'text': s.text} for s in segments]\nreport['asr'] = {'model': 'faster-whisper-medium', 'segments': observed,\n                 'note': 'ASR can make mistakes on sung lyrics; not a quality score.'}\n(root / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint('ASR_RESULT', json.dumps(observed, ensure_ascii=False), flush=True)\n"
checked = subprocess.run(["/content/ACE-Step-1.5/.venv/bin/python", "-u", "-c", CHECK], capture_output=True, text=True, timeout=600)
print(checked.stdout)
print(checked.stderr[-3000:])
assert checked.returncode == 0, "ASR verification failed; audio is retained."



Traceback (most recent call last):
  File "<string>", line 9, in <module>
  File "/content/ACE-Step-1.5/.venv/lib/python3.12/site-packages/faster_whisper/transcribe.py", line 876, in transcribe
    audio = decode_audio(audio, sampling_rate=sampling_rate)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ACE-Step-1.5/.venv/lib/python3.12/site-packages/faster_whisper/audio.py", line 46, in decode_audio
    with av.open(input_file, mode="r", metadata_errors="ignore") as container:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "av/container/core.py", line 479, in av.container.core.open
TypeError: open() got an unexpected keyword argument 'metadata_errors'



AssertionError: ASR verification failed; audio is retained.